# V2

Here We will use
NN Module's LOSS functions rather using Our own created Loss Function

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
print(df.head())


df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:, 0])

print(X_train.head())


# SCALLLING
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



# LABELING
label = LabelEncoder()
y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

print(y_test)



X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)



bias = torch.rand(1, dtype=torch.float64, requires_grad= True)

X_train_tensor = X_train_tensor.to(torch.float32)
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

In [32]:
learning_rate = 0.1
epochs = 45

In [33]:
class model_V2(nn.Module):

    def __init__(self, num_features):


        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):

        out = self.linear(features)
        out = self.sigmoid(out)

        return out
    
    


# Making/ Calling the Loss function

In [34]:
loss_fun = nn.BCELoss()

In [ ]:
model = model_V2(X_train_tensor.shape[1])

for epoch in range(epochs):
    # forward Pass
    y_pred = model(X_train_tensor)


    # loss calculate
    # loss = loss_fun(y_pred, y_train_tensor)  
    """thi will give error as--> 
    Using a target size (torch.Size([426])) that is different to the input size (torch.Size([426, 1])) is deprecated."""

    # we will do
    loss = loss_fun(y_pred, y_train_tensor.view(-1,1))


    print (f"{epoch} and loss {loss.item()}")

    # Backward pass
    loss.backward()
    
    # parametes Update / Optimization
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad       # we are doing GRADINT DECENT ---> Wnew = Wold - learingrate(dL/dW)
        model.linear.bias -= learning_rate * model.linear.bias.grad
    
    # Zero Grads
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()

0 and loss 0.6844534873962402
1 and loss 0.5203622579574585
2 and loss 0.43286046385765076
3 and loss 0.3785214424133301
4 and loss 0.34115782380104065
5 and loss 0.3136516809463501
6 and loss 0.2924002408981323
7 and loss 0.2753832936286926
8 and loss 0.26137810945510864
9 and loss 0.2495993971824646
10 and loss 0.2395186424255371
11 and loss 0.23076610267162323
12 and loss 0.22307486832141876
13 and loss 0.21624703705310822
14 and loss 0.21013252437114716
15 and loss 0.20461514592170715
16 and loss 0.19960355758666992
17 and loss 0.19502468407154083
18 and loss 0.19081947207450867
19 and loss 0.18693949282169342
20 and loss 0.18334466218948364
21 and loss 0.18000151216983795
22 and loss 0.17688177525997162
23 and loss 0.17396147549152374
24 and loss 0.1712200939655304
25 and loss 0.1686399281024933
26 and loss 0.16620562970638275
27 and loss 0.1639038622379303
28 and loss 0.16172295808792114
29 and loss 0.15965256094932556
30 and loss 0.1576835960149765
31 and loss 0.1558079570531845

In [37]:
# Model Evaluation

y_pred = y_pred.float()
with torch.no_grad():                     # Disable gradient calculation

    y_pred = model(X_test_tensor.float()) # Predict probabilities

    y_pred = (y_pred > 0.5).float()       # Convert probabilities to 0 or 1

    accuracy = (y_pred == y_test_tensor).float().mean()  # Calculate accuracy

    print(f"Accuracy: {accuracy.item()}") # Print accuracy

Accuracy: 0.5495133996009827
